# Combined Recommendation System

combining multiple recommenders with weighted scores. easy to add new ones later

In [1]:
%load_ext autoreload
%autoreload 2

## setup and data loading

first connect to db and load all the data we need

In [2]:
import os
import asyncio
import asyncpg
from dotenv import load_dotenv
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse

load_dotenv()
DATABASE_URL = os.getenv('DATABASE_URL')
if (DATABASE_URL is None):
    raise ValueError("DATABASE_URL is not set in environment variables")


def remove_schema_param(url):
    """Remove the schema parameter from DATABASE_URL for asyncpg"""
    parsed = urlparse(url)
    query_params = parse_qs(parsed.query)

    # Remove 'schema' parameter if it exists
    query_params.pop('schema', None)

    # Rebuild the URL without schema parameter
    new_query = urlencode(query_params, doseq=True)
    new_parsed = parsed._replace(query=new_query)
    return urlunparse(new_parsed)

asyncpg_url = remove_schema_param(DATABASE_URL)
conn = await asyncpg.connect(dsn=asyncpg_url)

## topic-based recommender setup

load the FAISS index and embeddings for finding similar events

In [3]:
import pandas as pd
import faiss
import numpy as np
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
import pickle
import math

from utils import load_events_df, load_user_event_associations


index_path = "event_embeddings_shard_"
meta_path = "models/event_index_meta.pkl"
model_name_label = "all-MiniLM-L6-v2"

user_event_associations_df = await load_user_event_associations(conn)

print(f"Loaded {len(user_event_associations_df):,} user-event rows")

# get combined events with caching
events_df = await load_events_df(conn)
print(f"Loaded {len(events_df)} rows into events_df")

shard_paths = sorted(
    os.path.join("models", p)
    for p in os.listdir("models")
    if p.startswith(index_path)
)

have_shards = len(shard_paths) > 0

if have_shards and os.path.exists(meta_path):
    print("Loading FAISS index and metadata from disk...")

    # load FAISS index
    indices = []
    shard_paths = sorted(p for p in os.listdir("models") if p.startswith("event_embeddings_shard_"))

    for p in shard_paths:
        indices.append(faiss.read_index(os.path.join("models", p)))

    d = indices[0].d
    index = faiss.IndexFlatIP(d)

    def get_vectors(idx: faiss.Index) -> np.ndarray:
        """Return all vectors stored in a flat FAISS index as (n, d) float32 array."""
        n = idx.ntotal
        if hasattr(idx, "reconstruct_n"):
            return idx.reconstruct_n(0, n)
        # fallback: reconstruct one by one if reconstruct_n is missing
        return np.vstack(idx.reconstruct(i) for i in range(n))

    for shard_id, idx in enumerate(indices):
        xb = get_vectors(idx).astype("float32")
        index.add(xb)
        print(f"Added shard {shard_id} with {idx.ntotal} vectors")

    print("Merged index ntotal:", index.ntotal)

    # load mappings / metadata
    with open(meta_path, "rb") as f:
        meta = pickle.load(f)

    index_to_id = meta["index_to_id"]
    id_to_index = meta["id_to_index"]
    model_name = meta.get("model_name", model_name_label)

    # reload embedding model
    embedding_model = SentenceTransformer(model_name)

    if hasattr(index, "reconstruct_n"):
        normalized_embeddings = index.reconstruct_n(0, index.ntotal)  # shape (n, d)
    else:
        # slower fallback
        normalized_embeddings = np.vstack(index.reconstruct(i) for i in range(n))

    print(normalized_embeddings.shape)
else:
    print("No saved index found. Building FAISS index from scratch...")
    topic_model = BERTopic(embedding_model=model_name_label)

    # Create combined text for embeddings
    events_df["combined_text"] = (
        events_df["title"].fillna("") + " " + events_df["description"].fillna("")
    )

    embedding_model = SentenceTransformer(model_name_label)
    embeddings = embedding_model.encode(
        events_df["combined_text"].tolist(), show_progress_bar=True
    )

    embedding_dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(embedding_dim)

    normalized_embeddings = embeddings / np.linalg.norm(
        embeddings, axis=1, keepdims=True
    )
    index.add(normalized_embeddings.astype("float32"))

    index_to_id = {i: id for i, id in enumerate(events_df.index)}
    id_to_index = {id: i for i, id in enumerate(events_df.index)}

    # Save FAISS index
    emb = normalized_embeddings.astype("float32")
    n, d = emb.shape
    shard_size = 15000  # adjust to keep each file <100MB

    n_shards = math.ceil(n / shard_size)

    for shard_id in range(n_shards):
        start = shard_id * shard_size
        end = min(start + shard_size, n)
        part = emb[start:end]

        idx = faiss.IndexFlatIP(d)
        idx.add(part)

        faiss.write_index(idx, f"models/event_embeddings_shard_{shard_id:02d}.index")

    # Save mappings (and any other metadata you like)
    meta = {
        "index_to_id": index_to_id,
        "id_to_index": id_to_index,
        "model_name": model_name_label,
    }

    with open(meta_path, "wb") as f:
        pickle.dump(meta, f)

c:\Users\euseb\anaconda3\envs\tools\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading user_event_associations_df from 14 cached files...
Loaded 483,767 user-event rows
Loading events_df from 3 cached parquet files in 'events_df_cache/'
Loaded 98450 rows into events_df
Loading FAISS index and metadata from disk...
Added shard 0 with 15000 vectors
Added shard 1 with 15000 vectors
Added shard 2 with 15000 vectors
Added shard 3 with 15000 vectors
Added shard 4 with 15000 vectors
Added shard 5 with 15000 vectors
Added shard 6 with 8450 vectors
Merged index ntotal: 98450
(98450, 384)


## cf setup

In [4]:
from collaborative_filtering.cf_recommender import load_cf_model, get_cf_scores_for_events, get_cf_users, get_event_title

# load the model
load_cf_model()

loading cf model...


c:\Users\euseb\anaconda3\envs\tools\Lib\site-packages\implicit\cpu\als.py:95: RuntimeWarning: OpenBLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


Loading cached data from 82 parquet files in 'collaborative_filtering/df_trade_tags_cache/'...
  8579 users, 3666 tags
Loading events_df from 7 cached files in 'events_tag_cache/'...
  65056 events with tags
done!


In [5]:
# from topic_based_recommender.recommend_events_for_user_by_topic_based import recommend_events_for_user_by_topic_based
# from recommendation_system.main import get_recommendations_for_user

# def get_topic_scores(user_id):
#     """topic based - finds similar events to what user interacted with"""
#     recs = recommend_events_for_user_by_topic_based(
#         id_to_index, normalized_embeddings, index, index_to_id,
#         user_event_associations_df, events_df,
#         user_id, top_n=100, similar_per_event=5
#     )
#     # convert rank to score
#     scores = {}
#     for i, r in enumerate(recs):
#         scores[r['id']] = 1.0 - (i / len(recs)) if recs else 0
#     return scores

# def get_cf_scores(user_id):
#     """collaborative filtering - based on similar users' preferences"""
#     return get_cf_scores_for_events(user_id)

# def get_association_scores(user_id):
#     """association rules """
#     return get_recommendations_for_user(user_id)

# RECOMMENDERS = {
#     'topic': (get_topic_scores, 0.3),
#     'cf': (get_cf_scores, 0.4),
#     'association': (get_association_scores, 0.3)
# }

# print(f"registered {len(RECOMMENDERS)} recommenders: {list(RECOMMENDERS.keys())}")

## combined recommender

each recommender is just a function that takes user_id and returns {event_id: score}
then we normalize and combine with weights. easy to add more recommenders later

In [6]:
# def normalize(scores):
#     if not scores: return {}
#     lo, hi = min(scores.values()), max(scores.values())
#     if hi == lo: return {k: 0.5 for k in scores}
#     return {k: (v-lo)/(hi-lo) for k, v in scores.items()}


# def get_recommendations(user_id, n=10):
#     """
#     combine all recommenders with their weights
#     """
#     # get scores from each recommender
#     all_scores = {}  # {name: {event_id: score}}
#     for name, (func, weight) in RECOMMENDERS.items():
#         try:
#             raw = func(user_id)
#             all_scores[name] = normalize(raw)
#         except Exception as e:
#             print(f"  warning: {name} failed for user {user_id[:10]}... ({e})")
#             all_scores[name] = {}
    
#     # normalize weights 
#     total_w = sum(w for _, w in RECOMMENDERS.values())
#     weights = {name: w/total_w for name, (_, w) in RECOMMENDERS.items()}
    
#     # combine scores
#     all_events = set()
#     for scores in all_scores.values():
#         all_events.update(scores.keys())
    
#     combined = {}
#     for eid in all_events:
#         combined[eid] = sum(
#             all_scores[name].get(eid, 0) * weights[name]
#             for name in RECOMMENDERS
#         )
    
#     # sort and return top n
#     top = sorted(combined.items(), key=lambda x: -x[1])[:n]
    
#     results = []
#     for eid, score in top:
#         # get title
#         if eid in events_df.index:
#             title = events_df.loc[eid, 'title']
#         else:
#             title = get_event_title(eid) or f"Event {eid}"
        
#         # also include individual scores for debugging
#         results.append({
#             'id': eid,
#             'title': title,
#             'score': score,
#             **{f'{name}_score': all_scores[name].get(eid, 0) for name in RECOMMENDERS}
#         })
    
#     return results

In [7]:
# # quick test
# test_users = list(set(user_event_associations_df['address'].unique()) & get_cf_users())
# print(f"{len(test_users)} users available")

# # pick one
# user = test_users[42]
# print(f"\ntesting: {user[:20]}...\n")

# recs = get_recommendations(user, n=10)

# # print results
# for i, r in enumerate(recs, 1):
#     t = r.get('topic_score', 0)
#     c = r.get('cf_score', 0)
#     title = str(r['title'])[:45] if isinstance(r['title'], str) else str(r['title'].iloc[0])[:45]
#     print(f"{i:2}. [{r['score']:.2f}] (t={t:.2f} c={c:.2f}) {title}")

## evaluation

see if our recs actually match what users bet on later (80/20 temporal split)

In [8]:
# from tqdm.auto import tqdm

# # prep test data
# print("prepping eval data...")
# eval_data = []

# for addr in tqdm(test_users[:500]):  # sample 500
#     evts = user_event_associations_df[
#         user_event_associations_df['address'] == addr
#     ]['event_id'].astype(int).unique()
    
#     if len(evts) < 5: continue
    
#     split = int(len(evts) * 0.8)
#     eval_data.append((addr, set(evts[split:])))  # just keep test set

# print(f"got {len(eval_data)} users")

# # DEBUG: check overlap between test events and recommendable events
# all_test_evts = set()
# for _, evts in eval_data:
#     all_test_evts.update(evts)

# # what can we actually recommend?
# sample_recs = get_recommendations(test_users[0], n=1000)  # get a lot
# recommendable = {r['id'] for r in sample_recs}

# overlap = all_test_evts & recommendable
# print(f"\ntest events: {len(all_test_evts)}")
# print(f"recommendable events: {len(recommendable)}")
# print(f"overlap: {len(overlap)}  <- this is why hit rate is 0!")

In [9]:
# # run eval
# hits = 0
# total_prec = 0
# k = 10

# for addr, test_evts in tqdm(eval_data, desc="evaluating"):
#     try:
#         recs = get_recommendations(addr, n=k)
#         rec_ids = {r['id'] for r in recs}
        
#         n_hit = len(rec_ids & test_evts)
#         total_prec += n_hit / k
#         if n_hit > 0:
#             hits += 1
#     except:
#         pass

# n = len(eval_data)
# print(f"hit rate@{k}: {hits/n:.4f}")
# print(f"precision@{k}: {total_prec/n:.4f}")

# print(f"evaluated on {n} users")

In [14]:
from topic_based_recommender.recommend_events_for_user_by_topic_based import recommend_events_for_user_by_topic_based
from collaborative_filtering.cf_recommender import load_cf_model, get_cf_scores_for_events, get_cf_users, get_event_title
#from recommendation_system.main import get_recommendations_for_user

def combined_recommendation_for_user(user_address):
    # This is the topic based recommender
    top_n=10
    similar_per_event=5
    cf_based_reccomender = get_cf_scores_for_events(user_address)
    #association_based_recommendations = get_recommendations_for_user(user_address)
    topic_based_recommendations= recommend_events_for_user_by_topic_based(id_to_index, normalized_embeddings,index,index_to_id, user_event_associations_df,events_df, user_address, top_n, similar_per_event)
    return topic_based_recommendations, cf_based_reccomender#, association_based_recommendations

In [15]:
example_user = user_event_associations_df.iloc[0]['address']
recommendations= combined_recommendation_for_user(example_user)
if recommendations:
    for rank, rec in enumerate(recommendations, 1):
        print(f"{rank}. [{rec['id']}] {rec['title']}")

TypeError: list indices must be integers or slices, not str